In [49]:
import sys
from pathlib import Path

# same reason as in 1-processing
sys.path.append(str(Path("..").resolve()))

# nest_asnycio instance to be able to call stan.build inside running event loop (Jupyter Notebooks)
import nest_asyncio

nest_asyncio.apply()

import pickle
import stan
from helpers import ModelData
import numpy as np

# Load processed data
data_path = "../data/processed/processed_data.pkl"
model_data = ModelData.from_pickle(data_path)

print(model_data.summary())


ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 200
- Eval indices: 221

Benchmark data: Available
- Benchmark covariates shape: (921, 23)
- Columns: business_id, density, Checkin, category, chain...



In [50]:
# Konfiguration für Stan-Modelle
stan_model_folder = Path("../data/stan_code")
fitted_model_folder = Path("../models")
fitted_model_folder.mkdir(parents=True, exist_ok=True)

# Cache für kompilierte Modelle (beschleunigt das Training)
# In PyStan 3 gibt es kein separates StanModel-Objekt mehr,
# daher speichern wir den model_code für jedes Modell
model_code_cache = {}

print(f"✓ Configuration set")
print(f"  Stan models: {stan_model_folder}")
print(f"  Output folder: {fitted_model_folder}")

✓ Configuration set
  Stan models: ../data/stan_code
  Output folder: ../models


In [51]:
# Funktion zum Konvertieren von ModelData zu Stan-Format
def prepare_stan_data(model_data: ModelData, S: int):
    """
    Bereitet die Daten für Stan vor und fügt die Anzahl der States hinzu.

    Parameters:
    -----------
    model_data : ModelData
        ModelData Objekt mit allen Daten
    S : int
        Anzahl der Hidden States

    Returns:
    --------
    dict : Dictionary im Stan-Format
    """
    # Stan erwartet spezifische Variablennamen (Großbuchstaben)
    stan_data = {
        "S": S,
        "N_total": model_data.n_total,
        "N_train": model_data.n_train,
        "N_obs": model_data.n_obs,
        "nCovs": model_data.n_covs,
        "Time": model_data.time,
        "Closed": model_data.closed.astype(int).tolist(),
        "Days": model_data.days,
        "Ratings": model_data.ratings,
        "Sentiment": model_data.sentiment,
        "Q": model_data.Q.tolist(),
        "R": model_data.R.tolist(),
        "X_test": model_data.X_test.tolist(),
    }

    return stan_data


# Test mit S=3
test_data = prepare_stan_data(model_data, S=3)
print(f"✓ Stan data prepared")
print(f"  Keys: {list(test_data.keys())}")

✓ Stan data prepared
  Keys: ['S', 'N_total', 'N_train', 'N_obs', 'nCovs', 'Time', 'Closed', 'Days', 'Ratings', 'Sentiment', 'Q', 'R', 'X_test']


In [58]:
# Funktion zum Trainieren eines einzelnen Modells mit PyStan 3
def train_model(
    model_data,
    S,
    model_name="vdhmm",
    num_chains=2,
    num_samples=1000,
    num_warmup=None,
    seed=None,
):
    """
    Trainiert ein VD-HMM oder HMM Modell mit PyStan 3.

    Parameters:
    -----------
    model_data : ModelData
        ModelData Objekt mit den Daten
    S : int
        Anzahl der Hidden States (1-4)
    model_name : str
        'vdhmm' oder 'hmm'
    num_chains : int
        Anzahl der MCMC Chains
    num_samples : int
        Anzahl der Post-Warmup Samples pro Chain
    num_warmup : int, optional
        Anzahl der Warmup Iterationen (default: num_samples)
    seed : int, optional
        Random seed für Reproduzierbarkeit

    Returns:
    --------
    stan.fit.Fit : Das trainierte Modell (Fit-Objekt)
    """

    assert model_name in [
        "vdhmm",
        "hmm",
    ], f"Invalid model_name '{model_name}'. Must be 'vdhmm' or 'hmm'."

    assert S in range(
        2, 5 + 1
    ), "Number of states must be in the range between 2 and 5!"

    # Daten vorbereiten
    stan_data = prepare_stan_data(model_data, S)

    model_file = stan_model_folder / f"{model_name}.stan"

    if not model_file.exists():
        raise FileNotFoundError(f"Stan model file not found: {model_file}")

    print(f"\n{'='*60}")
    print(f"Training {model_name.upper()} with S={S} states")
    print(f"{'='*60}")
    print(f"Model file: {model_file}")

    # Warmup setzen (PyStan 3 default ist num_samples wenn nicht angegeben)
    if num_warmup is None:
        num_warmup = num_samples

    # Model Code laden (mit Caching)
    model_key = f"{model_name}_{S}"
    if model_key not in model_code_cache:
        print(f"Loading Stan model code...")
        with open(model_file, "r") as f:
            model_code_cache[model_key] = f.read()
    else:
        print("Using cached model code")

    model_code = model_code_cache[model_key]

    # MCMC Sampling mit PyStan 3 API
    print(f"\nBuilding and sampling model:")
    print(f"  Chains: {num_chains}")
    print(f"  Warmup iterations: {num_warmup}")
    print(f"  Sampling iterations: {num_samples}")
    print(f"  Seed: {seed if seed else 'random'}")

    # stan.build() kompiliert das Modell mit Daten und seed
    # Wichtig: In PyStan 3 wird data und random_seed bei build() übergeben
    posterior = stan.build(program_code=model_code, data=stan_data, random_seed=seed)

    # posterior.sample() zieht Samples

    init_values = [{}] * num_chains

    fit = posterior.sample(
        num_chains=num_chains,
        num_samples=num_samples,
        num_warmup=num_warmup,
        init=init_values,
    )

    # Modell speichern
    output_path = fitted_model_folder / f"{model_name}_{S}.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(
            {
                "fit": fit,
                "model_name": model_name,
                "S": S,
                "stan_data": stan_data,
                "model_code": model_code,
            },
            f,
        )

    print(f"\n✓ Model saved to {output_path}")

    # Diagnostics ausgeben
    print(f"\n{'-'*60}")
    print("Model Summary:")
    print(f"{'-'*60}")

    # In PyStan 3: Zugriff auf Parameter via fit['param_name']
    # Konvertiere zu DataFrame für schönere Ausgabe
    df = fit.to_frame()
    print(f"\nSampled parameters: {df.shape[1]} parameters, {df.shape[0]} draws")
    print(f"Parameter names (first 10): {list(df.columns[:10])}")

    # Basis-Statistiken
    print(f"\nBasic statistics:")
    print(df.describe())

    return fit


print("✓ Training function defined (PyStan 3 API)")

✓ Training function defined (PyStan 3 API)


## Training VD-HMM Models (S=2 bis S=4)

Variable-Duration Hidden Markov Models mit zeitabhängigen Übergangswahrscheinlichkeiten.


In [59]:
# Training Settings
SEED = 42  # Für Reproduzierbarkeit
NUM_CHAINS = 2
NUM_SAMPLES = 1000  # Post-warmup samples pro chain
NUM_WARMUP = 1000  # Warmup iterations pro chain

np.random.seed(42)

# Dictionary zum Speichern aller trainierten Modelle
trained_models = {}

print("Training Configuration:")
print(f"  Seed: {SEED}")
print(f"  Chains: {NUM_CHAINS}")
print(f"  Samples (post-warmup): {NUM_SAMPLES}")
print(f"  Warmup iterations: {NUM_WARMUP}")
print(f"  Total iterations per chain: {NUM_WARMUP + NUM_SAMPLES}")

Training Configuration:
  Seed: 42
  Chains: 2
  Samples (post-warmup): 1000
  Warmup iterations: 1000
  Total iterations per chain: 2000


In [60]:
# VD-HMM Training für S=2 bis S=4
# WARNUNG: Dies kann mehrere Stunden dauern!

for S in range(2, 4 + 1):
    try:
        print(f"\n\n{'#'*60}")
        print(f"# VD-HMM Training: S={S}")
        print(f"{'#'*60}\n")

        fit = train_model(
            model_data=model_data,
            S=S,
            model_name="vdhmm",
            num_chains=NUM_CHAINS,
            num_samples=NUM_SAMPLES,
            num_warmup=NUM_WARMUP,
            seed=SEED,
        )

        trained_models[f"vdhmm_{S}"] = fit
        print(f"\n✓✓✓ VD-HMM with S={S} completed successfully! ✓✓✓\n")

    except Exception as e:
        print(f"\n✗✗✗ Error training VD-HMM with S={S}: {e} ✗✗✗\n")
        raise

print("\n" + "=" * 60)
print("VD-HMM Training Complete!")
print("=" * 60)



############################################################
# VD-HMM Training: S=2
############################################################


Training VDHMM with S=2 states
Model file: ../data/stan_code/vdhmm.stan
Using cached model code

Building and sampling model:
  Chains: 2
  Warmup iterations: 1000
  Sampling iterations: 1000
  Seed: 42
Building...



Building: found in cache, done.Messages from stanc:
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_smddt2s4/model_ysgp2rsx.stan', line 178, column 41: The
    variable state_emission may not have been assigned a value before its
    use.
Warning in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_smddt2s4/model_ysgp2rsx.stan', line 132, column 33: The
    variable pos_obs may not have been assigned a value before its use.
    provided, or the prior(s) depend on data variables. In the later case,
    this may be a false positive.
Sampling:   0%


✗✗✗ Error training VD-HMM with S=2: Exception during call to services function: `RuntimeError("Unrecoverable error evaluating the log probability at the initial value. Exception: vector[min_max] min indexing: accessing element out of range. index 2 out of range; expecting index to be between 1 and 1 (in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_rk_gn5s6/model_ysgp2rsx.stan', line 181, column 6 to column 45) ")`, traceback: `['  File "/opt/homebrew/Cellar/python@3.11/3.11.14_1/Frameworks/Python.framework/Versions/3.11/lib/python3.11/asyncio/tasks.py", line 277, in __step\n    result = coro.send(None)\n             ^^^^^^^^^^^^^^^\n', '  File "/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/httpstan/services_stub.py", line 182, in call\n    raise RuntimeError(exception_message)\n']` ✗✗✗



RuntimeError: Exception during call to services function: `RuntimeError("Unrecoverable error evaluating the log probability at the initial value. Exception: vector[min_max] min indexing: accessing element out of range. index 2 out of range; expecting index to be between 1 and 1 (in '/var/folders/hc/g9ymkbv17hbdwc11nytysx7m0000gn/T/httpstan_rk_gn5s6/model_ysgp2rsx.stan', line 181, column 6 to column 45) ")`, traceback: `['  File "/opt/homebrew/Cellar/python@3.11/3.11.14_1/Frameworks/Python.framework/Versions/3.11/lib/python3.11/asyncio/tasks.py", line 277, in __step\n    result = coro.send(None)\n             ^^^^^^^^^^^^^^^\n', '  File "/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.11/site-packages/httpstan/services_stub.py", line 182, in call\n    raise RuntimeError(exception_message)\n']`

## Training HMM Models (S=2 bis S=4)

Standard Hidden Markov Models mit konstanten Übergangswahrscheinlichkeiten.
(S=1 macht keinen Sinn für HMM und wird übersprungen)


In [ ]:
# HMM Training für S=2 bis S=4
# WARNUNG: Dies kann mehrere Stunden dauern!

for S in range(2, 4 + 1):
    try:
        print(f"\n\n{'#'*60}")
        print(f"# HMM Training: S={S}")
        print(f"{'#'*60}\n")

        fit = train_model(
            model_data=model_data,
            S=S,
            model_name="hmm",
            num_chains=NUM_CHAINS,
            num_samples=NUM_SAMPLES,
            num_warmup=NUM_WARMUP,
            seed=SEED,
        )

        trained_models[f"hmm_{S}"] = fit
        print(f"\n✓✓✓ HMM with S={S} completed successfully! ✓✓✓\n")

    except Exception as e:
        print(f"\n✗✗✗ Error training HMM with S={S}: {e} ✗✗✗\n")
        raise

print("\n" + "=" * 60)
print("HMM Training Complete!")
print("=" * 60)

## Training Summary

Übersicht über alle trainierten Modelle


In [ ]:
# Zusammenfassung aller trainierten Modelle
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"\nTotal models trained: {len(trained_models)}")
print(f"Models: {list(trained_models.keys())}")
print(f"\nSaved in: {fitted_model_folder}")

# Alle gespeicherten Modell-Dateien auflisten
saved_models = sorted(fitted_model_folder.glob("*.pkl"))
print(f"\nSaved model files ({len(saved_models)}):")
for model_file in saved_models:
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"  - {model_file.name} ({size_mb:.2f} MB)")

## Laden von trainierten Modellen

Falls du später ein bereits trainiertes Modell laden möchtest:


In [ ]:
# Beispiel: Lade ein trainiertes Modell
def load_fitted_model(model_name, S):
    """
    Lädt ein bereits trainiertes Modell aus dem Dateisystem.

    Parameters:
    -----------
    model_name : str
        'vdhmm' oder 'hmm'
    S : int
        Anzahl der States

    Returns:
    --------
    dict : Dictionary mit fit-Objekt und Metadaten
    """
    model_path = fitted_model_folder / f"{model_name}_{S}.pkl"

    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")

    with open(model_path, "rb") as f:
        model_dict = pickle.load(f)

    print(f"✓ Loaded {model_name.upper()} with S={S}")
    print(f"  Keys: {list(model_dict.keys())}")

    # In PyStan 3: fit ist ein stan.fit.Fit Objekt
    fit = model_dict["fit"]
    print(f"  Type: {type(fit)}")

    # Beispiel: Zugriff auf Parameter
    df = fit.to_frame()
    print(f"  Parameters: {df.shape[1]} params, {df.shape[0]} draws")

    return model_dict


# Beispiel: Lade VD-HMM mit S=3
# loaded_model = load_fitted_model('vdhmm', 3)
# fit = loaded_model['fit']
#
# # Zugriff auf spezifischen Parameter (PyStan 3 Syntax)
# if 'omega_0' in fit.to_frame().columns:
#     omega_0_samples = fit['omega_0']  # Direkter Zugriff via Dictionary-Syntax
#     print(f"omega_0 mean: {omega_0_samples.mean()}")